# Fase 2 — Preparação dos Dados (início)

**Tema 08:** RH e People Analytics  
**Objetivo desta etapa:** limpar a base, definir o que entra na clusterização e salvar artefatos prontos para a Fase 3.

**Entrada:** `data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv`  
**Saídas:**
- `data/processed/hr_limpo.csv` — base sem constantes
- `data/processed/hr_features_cluster.csv` — features para clusterização
- `data/processed/hr_avaliacao.csv` — ID + rótulos reservados para avaliação
- `data/processed/decisoes_preparacao.md` — registro das decisões

> Encoding/escala específicos de K-Means ficam para fechar na Aula 2/3, junto com Gower.  
> Gower trabalha bem com mistos **sem** one-hot obrigatório.


## 1. Carregar base

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT = DATA_PROCESSED
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_RAW / "WA_Fn-UseC_-HR-Employee-Attrition.csv")
print("Original:", df.shape)
df.head(2)

## 2. Remover colunas constantes

Conforme PDF do Tema 08 e a EDA: `EmployeeCount`, `StandardHours`, `Over18`.

In [ ]:
COLS_CONSTANTES = ["EmployeeCount", "StandardHours", "Over18"]

for c in COLS_CONSTANTES:
    assert df[c].nunique() == 1, f"{c} não é constante"

df_limpo = df.drop(columns=COLS_CONSTANTES)
print("Após remover constantes:", df_limpo.shape)
print("Removidas:", COLS_CONSTANTES)

## 3. Separar ID e rótulos de avaliação

- `EmployeeNumber` → identificador (não entra no cluster)
- `Attrition` → rótulo de retenção (**avaliação a posteriori**)
- `PerformanceRating` → desempenho (**avaliação a posteriori**; evita cluster "descobrir" o óbvio se o foco for perfil comportamental)

> Se a equipe decidir incluir Performance na clusterização, documente a mudança. Por padrão, reservamos.

In [ ]:
COL_ID = "EmployeeNumber"
COLS_AVALIACAO = ["Attrition", "PerformanceRating"]

df_avaliacao = df_limpo[[COL_ID] + COLS_AVALIACAO].copy()

df_cluster = df_limpo.drop(columns=[COL_ID] + COLS_AVALIACAO)

print("Avaliação:", df_avaliacao.shape)
print("Features cluster:", df_cluster.shape)
print("\nColunas para clusterização:")
print(df_cluster.columns.tolist())

## 4. Tipologia das variáveis (para Gower / encoding)

Três famílias:
1. **Categóricas nominais** (object)
2. **Likert / ordinais de satisfação**
3. **Numéricas contínuas / de contagem**

In [ ]:
LIKERT = [
    "EnvironmentSatisfaction",
    "JobSatisfaction",
    "RelationshipSatisfaction",
    "JobInvolvement",
    "WorkLifeBalance",
]

# Ordinais de cargo/educação/estoque (não são Likert de clima, mas também ordinais)
ORDINAIS = [
    "Education",
    "JobLevel",
    "StockOptionLevel",
]

CATEGORICAS = df_cluster.select_dtypes(include="object").columns.tolist()

NUMERICAS = [c for c in df_cluster.columns if c not in CATEGORICAS + LIKERT + ORDINAIS]

tipologia = {
    "categoricas_nominais": CATEGORICAS,
    "likert_clima": LIKERT,
    "ordinais": ORDINAIS,
    "numericas": NUMERICAS,
}

for k, v in tipologia.items():
    print(f"\n{k} ({len(v)}): {v}")

# sanity: todas as colunas classificadas, sem overlap indevido
todas = CATEGORICAS + LIKERT + ORDINAIS + NUMERICAS
assert (
    len(todas) == len(set(todas)) == df_cluster.shape[1]
), "Classificação incompleta ou com overlap"

## 5. Checagens finais de qualidade

In [ ]:
print("Ausentes em df_cluster:", int(df_cluster.isnull().sum().sum()))
print("Ausentes em df_avaliacao:", int(df_avaliacao.isnull().sum().sum()))
print("IDs únicos:", df_avaliacao[COL_ID].is_unique)

resumo_tipos = pd.DataFrame(
    {
        "coluna": df_cluster.columns,
        "dtype": df_cluster.dtypes.astype(str).values,
        "n_unicos": [df_cluster[c].nunique() for c in df_cluster.columns],
    }
)
resumo_tipos

## 6. (Opcional agora) Esboço de matriz para K-Means

Não é obrigatório para Gower. Útil se quiser comparar K-Means/GMM na Fase 3.

- One-hot nas categóricas nominais
- Manter Likert/ordinais como numéricas
- StandardScaler nas contínuas (aplicado na modelagem)

In [ ]:
df_kmeans_raw = pd.get_dummies(df_cluster, columns=CATEGORICAS, drop_first=False)
print("Matriz one-hot (ainda sem scaler):", df_kmeans_raw.shape)
df_kmeans_raw.head(2)

## 7. Salvar artefatos

In [ ]:
df_limpo.to_csv(OUT / "hr_limpo.csv", index=False)
df_cluster.to_csv(OUT / "hr_features_cluster.csv", index=False)
df_avaliacao.to_csv(OUT / "hr_avaliacao.csv", index=False)
df_kmeans_raw.to_csv(OUT / "hr_kmeans_raw_onehot.csv", index=False)

with open(OUT / "tipologia_variaveis.json", "w", encoding="utf-8") as f:
    json.dump(tipologia, f, ensure_ascii=False, indent=2)

print("Arquivos salvos em", OUT.resolve())
for p in sorted(OUT.glob("*")):
    print(" -", p.name)

In [ ]:
decisoes = f"""# Decisões de preparação — Tema 08 RH

**Data de geração:** automática via `03_preparacao.ipynb`

## Remoções
- Colunas constantes: `{', '.join(COLS_CONSTANTES)}`
- Motivo: variância zero (quebram padronização / não informam cluster)

## Fora da clusterização
- ID: `{COL_ID}`
- Avaliação a posteriori: `{', '.join(COLS_AVALIACAO)}`

## Tipologia para modelagem
- Categóricas nominais ({len(CATEGORICAS)}): {CATEGORICAS}
- Likert clima ({len(LIKERT)}): {LIKERT}
- Ordinais ({len(ORDINAIS)}): {ORDINAIS}
- Numéricas ({len(NUMERICAS)}): {NUMERICAS}

## Estratégia prevista (Fase 3)
1. **Principal:** distância de Gower + K-Medoids / Hierárquica / DBSCAN (k ≤ 5)
2. **Comparativo:** K-Means/GMM sobre matriz one-hot + scaler (`hr_kmeans_raw_onehot.csv`)
3. **PCA:** explorar redundância (JobLevel / MonthlyIncome / TotalWorkingYears)
4. **Associação:** Apriori/FP-Growth em variáveis discretizadas / flags

## Dimensões
- Original: {df.shape}
- Limpo: {df_limpo.shape}
- Features cluster: {df_cluster.shape}
- One-hot esboço: {df_kmeans_raw.shape}
"""

(OUT / "decisoes_preparacao.md").write_text(decisoes, encoding="utf-8")
print("decisoes_preparacao.md gravado")
print(decisoes)

## Checklist Fase 2 (início)

- [x] Remover colunas constantes
- [x] Separar ID
- [x] Reservar Attrition / PerformanceRating para avaliação
- [x] Classificar variáveis (nominal / Likert / ordinal / numérica)
- [x] Salvar CSVs processados + decisões
- [ ] Aplicar Gower (Fase 3)
- [ ] Padronizar + K-Means (Fase 3)
- [ ] Escolher k definitivo ≤ 5 (Fase 3)

**Pronto para Aula 2/3:** modelagem não supervisionada em `04_modelagem_clusters.ipynb` (a criar).